In [ ]:
import pandas as pd

# df_allnba = pd.read_csv('../data/processed/allnba_final.csv')
df_allnba = pd.read_csv('../data/processed/allnba_final_expanded.csv')

In [ ]:
df_allnba.shape

In [ ]:
path_voting_allnba = '../data/raw/voting_allnba.csv'
df_voting_allnba_raw = pd.read_csv(path_voting_allnba)
df_voting_allnba = df_voting_allnba_raw.copy()

In [ ]:
df_allnba['GP_RATIO'] = df_allnba['GP'] / 82  
df_allnba['MIN_PG'] = df_allnba['MIN'] / df_allnba['GP']

In [ ]:
df_allnba['PREV_YEAR_AWARD'] = df_allnba.groupby('PLAYER_ID')['TARGET'].shift(1).fillna(0)
df_allnba['CAREER_AWARDS']   = df_allnba.groupby('PLAYER_ID')['TARGET'].cumsum().shift(1).fillna(0)


In [ ]:
# Allnba
val_seasons = ['2022-23', '2023-24', '2024-25']

train_allnba = df_allnba[(df_allnba['SEASON'] >= '2018-19') & (df_allnba['SEASON'] <= '2021-22')]
val_allnba   = df_allnba[df_allnba['SEASON'].isin(val_seasons)]
test_allnba  = df_allnba[df_allnba['SEASON'] == '2025-26']

print(f'Train All-NBA: {train_allnba.shape}')
print(f'Val   All-NBA: {val_allnba.shape}')
print(f'Test  All-NBA: {test_allnba.shape}')


In [ ]:
meta_cols = ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'SEASON', 'PTS_WON', 'PTS_MAX']

# All-NBA
X_train_allnba = train_allnba.drop(columns=meta_cols + ['TARGET'])
y_train_allnba = train_allnba['TARGET']

X_val_allnba   = val_allnba.drop(columns=meta_cols + ['TARGET'])
y_val_allnba   = val_allnba['TARGET']

X_test_allnba  = test_allnba.drop(columns=meta_cols + ['TARGET'])
y_test_allnba  = test_allnba['TARGET']


In [ ]:
from unidecode import unidecode

def get_actual_teams(df_voting, season, n_teams):
    s = df_voting[df_voting['SEASON'] == season].copy()
    s = s.sort_values('Pts Won', ascending=False).reset_index(drop=True)
    
    teams = {}
    for i in range(1, n_teams + 1):
        start = (i - 1) * 5
        end = i * 5
        teams[i] = s.iloc[start:end]['Player'].apply(
            lambda x: unidecode(str(x).strip())
        ).tolist()
    return teams

def predict_teams_with_filter(model, X, df_meta, n_teams=3, team_size=5, min_games=0):
    df = df_meta.copy()
    df['PRED'] = model.predict(X)
    
    df.loc[df['GP'] < min_games, 'PRED'] = -999
    
    df_sorted = df.sort_values('PRED', ascending=False)
    teams = {}
    for i in range(1, n_teams + 1):
        start = (i - 1) * team_size
        end = i * team_size
        teams[i] = df_sorted.iloc[start:end]['PLAYER_NAME'].tolist()
    return teams


In [ ]:
def calculate_prediction_points(pred_teams, actual_teams):
    total_score = 0
    team_scores = {}  
    actual_team = {player: team_num for team_num, players in actual_teams.items() for player in players}

    for pred_team_num, pred_players in pred_teams.items():
        player_points = 0
        exact_matches = 0
        
        for player in pred_players:
            if player in actual_team:
                actual_num = actual_team[player]
                difference = abs(pred_team_num - actual_num)
                
                if difference == 0:
                    player_points += 10
                    exact_matches += 1
                elif difference == 1:
                    player_points += 8
                elif difference == 2:
                    player_points += 6
                    
        bonuses = {2: 5, 3: 10, 4: 20, 5: 40}
        
        team_total = player_points + bonuses.get(exact_matches, 0)
        team_scores[pred_team_num] = team_total
        
        total_score += team_total

    return total_score, team_scores


def models_test(models_dict, X_train, y_train, X_val, val_meta, voting_df, seasons, n_teams, min_games=0, show=True):
    if show:
        print(f"\n{'MODEL':<20} | {'SEZON':<10} | {'PUNKTY'}")
        print("-" * 50)
    
    final_results = {}

    for model_name, model in models_dict.items():
        model.fit(X_train, y_train)
        final_results[model_name] = {}
        model_total_score = 0
        
        for season in seasons:
            s_mask = val_meta['SEASON'] == season
            X_val_s = X_val[s_mask]
            
            meta_val_s = val_meta[s_mask][['PLAYER_NAME', 'SEASON', 'GP']]

            pred_teams = predict_teams_with_filter(model, X_val_s, meta_val_s, n_teams=n_teams, min_games=min_games)
            actual_teams = get_actual_teams(voting_df, season, n_teams=n_teams)
            
            total_points, team_scores = calculate_prediction_points(pred_teams, actual_teams)
            
            final_results[model_name][season] = total_points
            model_total_score += total_points
            
            # print(pred_teams)
            if show:
                print(f"{model_name:<20} | {season:<10} | Suma: {total_points}")
            
                for team_num, score in sorted(team_scores.items()):
                    print(f"{'':<20} | {'':<10} |  - {team_num} Team: {score} pkt")
        
        if show:
            print(f"{'':<20} | {'Średnia':<10} | {model_total_score/len(seasons):.2f}")
            print("-" * 50)
        
    return final_results

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, ndcg_score
import numpy as np

models_allnba = {
    # model_allnba_final
    'CatBoostRegressor_default': CatBoostRegressor(
        random_state=42,
        iterations=300, 
        learning_rate=0.1, 
        depth=3, 
        verbose=0
    ),

    # model_allnba_final_expanded
    'CatBoostRegressor_expanded': CatBoostRegressor(
        random_state=42,
        iterations=100, 
        learning_rate=0.01, 
        depth=3, 
        verbose=0
    )
}

In [ ]:
print("=== WYNIKI DLA ALL-NBA ===")
wyniki_allnba = models_test(
    models_dict=models_allnba, 
    X_train=X_train_allnba, 
    y_train=y_train_allnba, 
    X_val=X_val_allnba, 
    val_meta=val_allnba, 
    voting_df=df_voting_allnba, 
    seasons=val_seasons, 
    n_teams=3, 
    min_games=63
)

In [ ]:
from itertools import product
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth':    [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
}

results = []
for params in [dict(zip(param_grid, v)) for v in product(*param_grid.values())]:
    models = {
        'xgb': XGBRegressor(**params, random_state=42),
        'cat': CatBoostRegressor(**params, random_state=42, verbose=0),
    }
    
    wyniki = models_test(
        models_dict=models,
        X_train=X_train_allnba, y_train=y_train_allnba,
        X_val=X_val_allnba,     val_meta=val_allnba,
        voting_df=df_voting_allnba,
        seasons=val_seasons,    n_teams=3, min_games=63, show=False
    )
    print(f'Finished params: {params} with score: {wyniki}')
    
    for model_name, season_scores in wyniki.items():
        total = sum(season_scores.values())
        results.append({'model': model_name, **params, 'score': round(total/3, 2), 'detail_results': season_scores})

df_results = pd.DataFrame(results).sort_values('score', ascending=False)
print(df_results.head(10))